# Практика 8. Очистка данных и контроль качества

На практике 7 мы научились данные **загружать** и проверять, что при загрузке ничего не
потерялось. Сегодня следующий шаг: данные загружены полностью — и они плохие.

**Задача пары:** привести в порядок выгрузку рейсов за май 2025 и посчитать, **на сколько
изменится выручка** после очистки. Это не упражнение на методы: каждое решение по очистке
меняет итоговое число в отчёте.

| Часть | О чём |
|---|---|
| 1 | Аудит: карта проблем до начала работы |
| 2 | Дубликаты: полные, по ключу и конфликтующие |
| 3 | Преобразование типов |
| 4 | Нормализация значений |
| 5 | Пропуски: диагностика и стратегии |
| 6 | Некорректные наблюдения и аномалии |
| 7 | Мини-проект: пайплайн очистки и отчёт о качестве |

> **Главная мысль пары:** очистка данных — это последовательность **решений**, а не
> последовательность методов. `dropna()` и `fillna(0)` — одна строка кода каждая, но это
> разные ответы на вопрос «что считать правдой». Поэтому каждое решение нужно записывать:
> именно из них складывается итоговая цифра, которую вы положите на стол.

---
## Шаг 0. Проект

```
# терминал
mkdir data-cleaning
cd data-cleaning
```

`requirements.txt`:

```
pandas
```

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

`.gitignore` с `.venv/`, `git init`, kernel — `.venv`.

---
## Данные

Запустите ячейку — она создаст `data/trips_raw.csv`: выгрузку рейсов за май в том виде,
в каком её прислали. Посмотрите на первые строки глазами, прежде чем писать код.

| Колонка | Смысл |
|---|---|
| `trip_id` | номер рейса |
| `date` | дата рейса |
| `route` | маршрут |
| `carrier` | перевозчик |
| `cargo_type` | тип груза |
| `client` | клиент |
| `weight_t` | вес груза, тонн |
| `distance_km` | расстояние, км |
| `duration_h` | время в пути, часов |
| `cost_rub` | стоимость перевозки, руб. |
| `is_insured` | был ли груз застрахован |
| `delay_h` | задержка, часов |

In [172]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт data/trips_raw.csv — выгрузку рейсов за май 2025 "как она пришла".

import csv
import random
from pathlib import Path

random.seed(8)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
# у каждого перевозчика одно правильное имя и несколько написаний из разных систем
carrier_variants = {
    "ТрансЛогистик": ["ТрансЛогистик", "транслогистик", 'ООО "ТрансЛогистик"', "  ТрансЛогистик", "ТРАНСЛОГИСТИК"],
    "РусКарго": ["РусКарго", "рускарго", "ООО РусКарго", "РусКарго "],
    "СибПуть": ["СибПуть", "сибпуть", 'АО "СибПуть"'],
    "ВолгаТранс": ["ВолгаТранс", "волгатранс", "ВолгаТранс  "],
    "АвтоДорСервис": ["АвтоДорСервис", "автодорсервис", "ООО АвтоДорСервис"],
}
cargo_variants = {
    "Контейнер": ["Контейнер", "контейнер", "КОНТЕЙНЕР"],
    "Нефтепродукты": ["Нефтепродукты", "нефтепродукты"],
    "Зерно": ["Зерно", "зерно"],
    "Металл": ["Металл", "металл", "МЕТАЛЛ"],
    "Лес": ["Лес", "лес"],
    "Стройматериалы": ["Стройматериалы", "стройматериалы"],
}
tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}
clients = ["Агрохолдинг Нива", "МеталлПром", "СтройБаза 77", "НефтеТрейд", "ЛесЭкспорт", "РитейлСеть"]

rows = []
for i in range(1, 481):
    route, distance = random.choice(list(routes.items()))
    cargo_canon = random.choice(list(tariffs))
    carrier_canon = random.choice(list(carrier_variants))

    weight = round(random.uniform(3, 22), 1)
    speed = random.uniform(45, 65)
    duration = round(distance / speed, 1)
    cost = distance * weight * tariffs[cargo_canon] * random.uniform(0.95, 1.08)
    delay = 0 if random.random() < 0.7 else random.choice([1, 2, 3, 4, 6, 8])

    # дата приходит в двух разных форматах — системы выгружали по-своему
    day = random.randint(1, 31)
    if random.random() < 0.65:
        date = f"{day:02d}.05.2025"
    else:
        date = f"2025-05-{day:02d}"

    # признак страховки записан кто как хотел
    insured = random.choice(["да", "нет", "Да", "НЕТ", "1", "0", ""])

    row = {
        "trip_id": f"T{i:04d}",
        "date": date,
        "route": random.choice([route, f" {route} ", route]),
        "carrier": random.choice(carrier_variants[carrier_canon]),
        "cargo_type": random.choice(cargo_variants[cargo_canon]),
        "client": "" if random.random() < 0.03 else random.choice(clients),
        "weight_t": "" if random.random() < 0.06 else weight,
        "distance_km": distance,
        "duration_h": duration,
        # стоимость как текст: пробел в разрядах и десятичная запятая
        "cost_rub": f"{cost:,.2f}".replace(",", " ").replace(".", ","),
        "is_insured": insured,
        # пустое поле задержки означает "не заполнили", а не "задержки не было"
        "delay_h": "" if random.random() < 0.08 else delay,
    }

    r = random.random()
    if r < 0.017:                       # вес не мог быть нулевым или отрицательным
        row["weight_t"] = random.choice([0, -2.5, 0.0])
    elif r < 0.034:                     # такая скорость физически невозможна для фуры
        row["duration_h"] = round(distance / random.uniform(150, 400), 1)
    elif r < 0.044:                     # потерянный разряд: стоимость в 10 раз больше
        row["cost_rub"] = f"{cost * 10:,.2f}".replace(",", " ").replace(".", ",")
    elif r < 0.05:                      # дата из будущего
        row["date"] = f"{random.randint(1, 28):02d}.05.2026"

    rows.append(row)

# полные дубли строк: выгрузка запускалась дважды
rows += [dict(rows[i]) for i in (10, 55, 120, 200, 301, 333, 410, 455)]

# дубли по номеру рейса, но с разными суммами: рейс правили и выгрузили обе версии
for i in (20, 77, 150, 260, 390):
    twin = dict(rows[i])
    base = float(twin["cost_rub"].replace(" ", "").replace(",", "."))
    twin["cost_rub"] = f"{base * 1.12:,.2f}".replace(",", " ").replace(".", ",")
    rows.append(twin)

random.shuffle(rows)

Path("data").mkdir(exist_ok=True)
with open("data/trips_raw.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Готово. Создан файл data/trips_raw.csv")
print("Строк:", len(rows))
print("Колонки:", list(rows[0].keys()))
print()
print("Первые 5 строк файла как есть:")
for line in open("data/trips_raw.csv", encoding="utf-8").read().split("\n")[:6]:
    print("   ", line)

Готово. Создан файл data/trips_raw.csv
Строк: 493
Колонки: ['trip_id', 'date', 'route', 'carrier', 'cargo_type', 'client', 'weight_t', 'distance_km', 'duration_h', 'cost_rub', 'is_insured', 'delay_h']

Первые 5 строк файла как есть:
    trip_id,date,route,carrier,cargo_type,client,weight_t,distance_km,duration_h,cost_rub,is_insured,delay_h
    T0151,06.05.2025,Казань-Самара,ВолгаТранс  ,Стройматериалы,Агрохолдинг Нива,8.4,360,6.2,"17 325,35",Да,2
    T0104,2025-05-13,Екатеринбург-Челябинск,волгатранс,нефтепродукты,МеталлПром,16.4,220,4.6,"31 724,81",НЕТ,0
    T0041,2025-05-29,Москва-Санкт-Петербург,ВолгаТранс,Металл,ЛесЭкспорт,-2.5,710,13.2,"100 254,36",1,0
    T0358,2025-05-29,Москва-Нижний Новгород,рускарго,зерно,Агрохолдинг Нива,,420,7.4,"23 425,16",,0
    T0361,10.05.2025,Санкт-Петербург-Петрозаводск,ТрансЛогистик,нефтепродукты,НефтеТрейд,,430,8.1,"16 258,56",,0


---
# Часть 1. Аудит: сначала карта, потом чистка

Главная ошибка новичка — начать чистить сразу: увидел пропуск, заполнил; увидел дубликат,
удалил. В итоге непонятно, что было сделано и почему, а итоговые числа невозможно объяснить.

Правильный порядок: **сначала полный аудит, потом план, потом чистка**. Аудит — это пять команд.

| Команда | Что показывает |
|---|---|
| `df.shape` | сколько строк и столбцов |
| `df.dtypes` | какие типы определил pandas — и где он не справился |
| `df.isna().sum()` | пропуски по столбцам |
| `df.duplicated().sum()` | полные дубликаты строк |
| `df["col"].nunique()` | сколько разных значений в столбце |

Ключевой признак проблемы с типами: столбец, который **должен** быть числом или датой,
определился как `str` (в pandas 2.x — `object`).

In [173]:
import pandas as pd

raw = pd.read_csv("data/trips_raw.csv")

print("размер:", raw.shape)
print()
print(raw.dtypes)
print()
raw.head()

размер: (493, 12)

trip_id            str
date               str
route              str
carrier            str
cargo_type         str
client             str
weight_t       float64
distance_km      int64
duration_h     float64
cost_rub           str
is_insured         str
delay_h        float64
dtype: object



,trip_id,date,route,carrier,cargo_type,client,weight_t,distance_km,duration_h,cost_rub,is_insured,delay_h
0,T0151,06.05.2025,Казань-Самара,ВолгаТранс,Стройматериалы,Агрохолдинг Нива,8.4,360,6.2,"17 325,35",Да,2.0
1,T0104,2025-05-13,Екатеринбург-Челябинск,волгатранс,нефтепродукты,МеталлПром,16.4,220,4.6,"31 724,81",НЕТ,0.0
2,T0041,2025-05-29,Москва-Санкт-Петербург,ВолгаТранс,Металл,ЛесЭкспорт,-2.5,710,13.2,"100 254,36",1,0.0
3,T0358,2025-05-29,Москва-Нижний Новгород,рускарго,зерно,Агрохолдинг Нива,NaN,420,7.4,"23 425,16",NaN,0.0
4,T0361,10.05.2025,Санкт-Петербург-Петрозаводск,ТрансЛогистик,нефтепродукты,НефтеТрейд,NaN,430,8.1,"16 258,56",NaN,0.0


In [174]:
print("ПРОПУСКИ:")
na = raw.isna().sum()
print(na[na > 0])
print()

print("ДУБЛИКАТЫ:")
print("  полных дублей строк:", raw.duplicated().sum())
print("  дублей по trip_id:  ", raw.duplicated(subset="trip_id").sum())
print()

print("РАЗНООБРАЗИЕ ЗНАЧЕНИЙ (там, где его быть не должно):")
for col in ["carrier", "cargo_type", "route", "is_insured"]:
    print(f"  {col}: {raw[col].nunique()} разных значений")

ПРОПУСКИ:
client        13
weight_t      29
is_insured    63
delay_h       40
dtype: int64

ДУБЛИКАТЫ:
  полных дублей строк: 8
  дублей по trip_id:   13

РАЗНООБРАЗИЕ ЗНАЧЕНИЙ (там, где его быть не должно):
  carrier: 18 разных значений
  cargo_type: 14 разных значений
  route: 16 разных значений
  is_insured: 6 разных значений


Посмотрите на последний блок. Перевозчиков в компании пять, маршрутов восемь, типов груза
шесть — а в данных значений гораздо больше. Значит одно и то же записано по-разному.
Посмотрим, как именно.

In [175]:
print("ВАРИАНТЫ НАПИСАНИЯ ПЕРЕВОЗЧИКА:")
for value in sorted(raw["carrier"].unique()):
    print(f"  {value!r}")
print()
print("ВАРИАНТЫ ПРИЗНАКА СТРАХОВКИ:")
print(" ", raw["is_insured"].dropna().unique().tolist())

ВАРИАНТЫ НАПИСАНИЯ ПЕРЕВОЗЧИКА:
  '  ТрансЛогистик'
  'АО "СибПуть"'
  'АвтоДорСервис'
  'ВолгаТранс'
  'ВолгаТранс  '
  'ООО "ТрансЛогистик"'
  'ООО АвтоДорСервис'
  'ООО РусКарго'
  'РусКарго'
  'РусКарго '
  'СибПуть'
  'ТРАНСЛОГИСТИК'
  'ТрансЛогистик'
  'автодорсервис'
  'волгатранс'
  'рускарго'
  'сибпуть'
  'транслогистик'

ВАРИАНТЫ ПРИЗНАКА СТРАХОВКИ:
  ['Да', 'НЕТ', '1', 'да', '0', 'нет']


### Задание 1. Составьте карту проблем

Заполните таблицу числами из аудита. Это ваш план работы на пару: по каждой строке
потом нужно будет принять решение.

| Проблема | Сколько | Что решили делать |
|---|---|---|
| строк всего |493| — |
| полные дубликаты строк |8|удалить|
| дубликаты по `trip_id` |13|изучить конфликтующие строки и оставить правильную запись|
| пропуски в `weight_t` |29|заполнить средним арифметическим по типу груза|
| пропуски в `delay_h` |40|заполнить медианой|
| пропуски в `client` |13|поставить прочерки|
| пропуски в `is_insured` |63|поставить прочерки|
| столбцы с неверным типом |2|преобразовать тип данных (у "date" - datetime, у "cost_rub" - float)|
| вариантов написания `carrier` |18|привести к единому виду|
| вариантов написания `cargo_type` |14|привести к единому виду|

Отдельно ответьте: **какие столбцы pandas определил как текст, хотя они должны быть
числом или датой?** Почему он не справился?

In [176]:
print(len(raw))
print(raw.duplicated().sum())
print(raw.duplicated(subset="trip_id").sum())
print(raw["weight_t"].isna().sum())
print(raw["delay_h"].isna().sum())
print(raw["client"].isna().sum())
print(raw["is_insured"].isna().sum())
print(raw["carrier"].nunique())
print(raw["cargo_type"].nunique())
raw.dtypes
# pandas не смог праивльно определить тип у:
# 1.cost_rub (потому что десятичные числа записаны через ",", а не ".")
# 2.date (потому что данные записаны в разных форматах)

493
8
13
29
40
13
63
18
14


trip_id            str
date               str
route              str
carrier            str
cargo_type         str
client             str
weight_t       float64
distance_km      int64
duration_h     float64
cost_rub           str
is_insured         str
delay_h        float64
dtype: object

---
# Часть 2. Дубликаты

Дубликаты бывают трёх видов, и это три разные задачи.

| Вид | Как найти | Что делать |
|---|---|---|
| **полный дубль строки** | `df.duplicated()` | удалить: это технический сбой выгрузки |
| **дубль по ключу** (один `trip_id`, строки одинаковые) | `df.duplicated(subset="trip_id")` | удалить |
| **конфликтующий дубль** (один `trip_id`, значения **разные**) | группировка по ключу + `nunique()` | разобраться: какая версия верна |

Третий случай — самый опасный. Один и тот же рейс выгрузился дважды с разной стоимостью:
кто-то поправил сумму, и выгрузка отдала обе версии. Простое `drop_duplicates()` здесь
**не поможет**: строки не одинаковые, и оно оставит обе.

In [177]:
# 1. Полные дубли: все 12 значений строки совпадают
full_dups = raw[raw.duplicated(keep=False)]
print("строк, участвующих в полных дублях:", len(full_dups))
print(full_dups.sort_values("trip_id").head(4)[["trip_id", "date", "carrier", "cost_rub"]])
print()

# 2. Дубли по ключу
key_dups = raw[raw.duplicated(subset="trip_id", keep=False)]
print("строк с повторяющимся trip_id:", len(key_dups))
print("уникальных номеров рейсов:", raw["trip_id"].nunique(), "из", len(raw), "строк")

строк, участвующих в полных дублях: 16
    trip_id        date            carrier    cost_rub
100   T0011  08.05.2025      автодорсервис   35 667,91
247   T0011  08.05.2025      автодорсервис   35 667,91
313   T0056  2025-05-05  ООО АвтоДорСервис  101 324,13
329   T0056  2025-05-05  ООО АвтоДорСервис  101 324,13

строк с повторяющимся trip_id: 26
уникальных номеров рейсов: 480 из 493 строк


In [178]:
# 3. Конфликтующие дубли: один номер рейса, разная стоимость
conflicts = (key_dups.groupby("trip_id")["cost_rub"]
             .nunique()
             .loc[lambda s: s > 1])

print("номеров рейсов с расхождением в стоимости:", len(conflicts))
print(conflicts)
print()
print("пример расхождения:")
example_id = conflicts.index[0]
print(raw[raw["trip_id"] == example_id][["trip_id", "date", "carrier", "cost_rub"]])

номеров рейсов с расхождением в стоимости: 5
trip_id
T0021    2
T0078    2
T0151    2
T0261    2
T0391    2
Name: cost_rub, dtype: int64

пример расхождения:
    trip_id        date     carrier   cost_rub
214   T0021  2025-05-03  ВолгаТранс  32 466,20
338   T0021  2025-05-03  ВолгаТранс  28 987,68


### Задание 2

1. Удалите полные дубликаты. Сколько строк осталось?
2. Посмотрите на конфликтующие дубли (один `trip_id`, разные суммы). Решите, какую версию
   оставить, и **обоснуйте решение**. Варианты: оставить первую, оставить последнюю,
   оставить максимальную сумму, выяснить у источника.
3. Примените своё решение и получите таблицу, где каждый `trip_id` встречается один раз.
   Проверьте это кодом.
4. Запишите, на какую сумму изменилась общая стоимость рейсов после удаления дублей.
   (Стоимость пока текстовая — посчитайте после части 3 или просто вернитесь к этому пункту.)

> Подсказка к пункту 3: у `drop_duplicates` есть параметр `keep` со значениями
> `"first"`, `"last"` и `False` (удалить вообще все повторяющиеся). Для «оставить
> максимальную» пригодится сортировка перед удалением.

In [179]:
# 1)
df = raw.drop_duplicates()
print("после удаления полных дублей:", len(df))

# 2) ваше решение и обоснование — в markdown-ячейке ниже
conflikt = (df.groupby("trip_id")["cost_rub"].nunique().loc[lambda s: s > 1])
print(conflikt)
display(df[df["trip_id"].isin(conflikt.index)].sort_values(by="trip_id"))
# 3)
df["correct_cost_rub"] = df["cost_rub"].str.replace(" ", "").str.replace(",", ".").astype(float)
df = df.sort_values(by=["trip_id", "cost_rub"], ascending=[True, False])
summa1 = df["correct_cost_rub"].sum()
df = df.drop_duplicates(subset="trip_id", keep="first")
conflikt = (df.groupby("trip_id")["cost_rub"].nunique().loc[lambda s: s > 1])
print("дублей по trip_id осталось:", len(conflikt))
#4
summa2 = df["correct_cost_rub"].sum()
summa = summa1 - summa2
print("Cумма изменилась на", summa, "рублей")

после удаления полных дублей: 485
trip_id
T0021    2
T0078    2
T0151    2
T0261    2
T0391    2
Name: cost_rub, dtype: int64


,trip_id,date,route,carrier,cargo_type,client,weight_t,distance_km,duration_h,cost_rub,is_insured,delay_h
214,T0021,2025-05-03,Казань-Самара,ВолгаТранс,зерно,ЛесЭкспорт,16.3,360,6.3,"32 466,20",НЕТ,0.0
338,T0021,2025-05-03,Казань-Самара,ВолгаТранс,зерно,ЛесЭкспорт,16.3,360,6.3,"28 987,68",НЕТ,0.0
50,T0078,2025-05-13,Казань-Самара,транслогистик,Стройматериалы,НефтеТрейд,4.5,360,7.4,"10 828,54",да,0.0
60,T0078,2025-05-13,Казань-Самара,транслогистик,Стройматериалы,НефтеТрейд,4.5,360,7.4,"9 668,34",да,0.0
0,T0151,06.05.2025,Казань-Самара,ВолгаТранс,Стройматериалы,Агрохолдинг Нива,8.4,360,6.2,"17 325,35",Да,2.0
112,T0151,06.05.2025,Казань-Самара,ВолгаТранс,Стройматериалы,Агрохолдинг Нива,8.4,360,6.2,"19 404,39",Да,2.0
118,T0261,10.05.2025,Москва-Нижний Новгород,ТРАНСЛОГИСТИК,Металл,Агрохолдинг Нива,5.9,420,6.6,"17 134,72",1,3.0
460,T0261,10.05.2025,Москва-Нижний Новгород,ТРАНСЛОГИСТИК,Металл,Агрохолдинг Нива,5.9,420,6.6,"19 190,89",1,3.0
191,T0391,2025-05-05,Санкт-Петербург-Петрозаводск,волгатранс,Контейнер,МеталлПром,11.7,430,8.3,"37 662,45",Да,0.0
227,T0391,2025-05-05,Санкт-Петербург-Петрозаводск,волгатранс,Контейнер,МеталлПром,11.7,430,8.3,"42 181,94",Да,0.0


дублей по trip_id осталось: 0
Cумма изменилась на 111938.73999999836 рублей


> **Решение по конфликтующим дублям:** если не можем связаться с источником, оставляем макисмальную, потому что та цена, что ниже, может оказаться заниженной, и компания потеряет деньги.

In [180]:
if "correct_cost_rub" in df.columns:
    df = df.drop(columns=["correct_cost_rub"])

---
# Часть 3. Преобразование типов

Три типовые задачи: число из текста, дата из текста, логическое значение из текста.

## 3.1 Число из текста

`cost_rub` пришёл как `45 320,50`: пробел-разделитель разрядов и десятичная запятая.
Чтобы получить число, нужно убрать пробелы и заменить запятую на точку.

Работаем через `.str` — это доступ к строковым методам всего столбца сразу.

> Отдельная тонкость: в выгрузках часто встречается **неразрывный пробел** (символ `\xa0`)
> вместо обычного. Выглядит одинаково, но `.str.replace(" ", "")` его не удалит. Если число
> упорно не преобразуется — проверьте это первым делом.

In [181]:
print("было:", repr(raw["cost_rub"].iloc[0]), "| тип:", raw["cost_rub"].dtype)

cost = (raw["cost_rub"].astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace("\u00a0", "", regex=False)      # неразрывный пробел
        .str.replace(",", ".", regex=False)
        .astype(float))

print("стало:", cost.iloc[0], "| тип:", cost.dtype)
print()
print("проверка: сумма посчиталась как число:", round(cost.sum(), 2))

было: '17 325,35' | тип: str
стало: 17325.35 | тип: float64

проверка: сумма посчиталась как число: 21045802.36


## 3.2 Дата из текста — и ловушка, которая стоит дороже всего

В столбце `date` **два формата одновременно**: `15.05.2025` и `2025-05-15`. Так бывает,
когда данные собраны из двух систем.

Напрашивается `pd.to_datetime(..., format="mixed", dayfirst=True)`. Запустите ячейку ниже
и посмотрите на результат внимательно.

In [182]:
naive = pd.to_datetime(raw["date"], format="mixed", dayfirst=True, errors="coerce")

print("ошибок при разборе:", naive.isna().sum())
print("диапазон дат:", naive.min().date(), "-", naive.max().date())
print()
print("а теперь посмотрим, в какие месяцы попали рейсы:")
print(naive.dt.month.value_counts().sort_index())

ошибок при разборе: 0
диапазон дат: 2025-01-05 - 2026-05-16

а теперь посмотрим, в какие месяцы попали рейсы:
date
1       8
2       6
3       5
4       7
5     436
6       4
7       4
8       6
9       5
10      7
11      1
12      4
Name: count, dtype: int64


Все рейсы — майские, но в результате появились январь, февраль, март и далее.
**Ошибок при разборе ноль.** Тип преобразовался, данные испортились, никто не предупредил.

Причина: `dayfirst=True` применяется ко **всем** строкам, включая `2025-05-01`.
Там день и месяц меняются местами, и 1 мая превращается в 5 января. Пострадали все даты,
у которых номер дня не больше 12 — то есть почти половина.

Правильный способ — разобрать каждый формат **отдельно** и склеить результат:

```python
d1 = pd.to_datetime(col, format="%d.%m.%Y", errors="coerce")   # то, что подошло — разобрано
d2 = pd.to_datetime(col, format="%Y-%m-%d", errors="coerce")   # остальное
result = d1.fillna(d2)                                         # склеиваем
```

`errors="coerce"` ставит `NaT` там, где формат не подошёл — именно это и позволяет
склеить два результата.

In [183]:
d1 = pd.to_datetime(raw["date"], format="%d.%m.%Y", errors="coerce")
d2 = pd.to_datetime(raw["date"], format="%Y-%m-%d", errors="coerce")
date_ok = d1.fillna(d2)

print("разобрано первым форматом:", d1.notna().sum())
print("разобрано вторым форматом:", d2.notna().sum())
print("не разобрано совсем:", date_ok.isna().sum())
print()
print("месяцы теперь:")
print(date_ok.dt.month.value_counts().sort_index())
print()
print("диапазон:", date_ok.min().date(), "-", date_ok.max().date())

разобрано первым форматом: 316
разобрано вторым форматом: 177
не разобрано совсем: 0

месяцы теперь:
date
5    493
Name: count, dtype: int64

диапазон: 2025-05-01 - 2026-05-16


## 3.3 Логическое значение из текста

`is_insured` записан как `да`, `нет`, `Да`, `НЕТ`, `1`, `0` и пустое значение.
Нужен один тип — `bool`. Надёжный способ: привести к нижнему регистру и подставить
по словарю через `.map()`.

Важно: `.map()` поставит `NaN` для всего, чего нет в словаре. Это не баг, а способ
**увидеть** неучтённые варианты — обязательно проверяйте результат.

In [184]:
insured_map = {"да": True, "1": True, "нет": False, "0": False}

insured = (raw["is_insured"].astype(str).str.strip().str.lower()
           .map(insured_map))

print("True :", (insured == True).sum())
print("False:", (insured == False).sum())
print("пропусков:", insured.isna().sum(), "- это строки, где признак не был заполнен")
print()
# контроль: не потеряли ли мы вариант, которого нет в словаре
unmapped = raw.loc[insured.isna() & raw["is_insured"].notna(), "is_insured"].unique()
print("значения, не попавшие в словарь:", unmapped)

True : 219
False: 211
пропусков: 63 - это строки, где признак не был заполнен

значения, не попавшие в словарь: <StringArray>
[]
Length: 0, dtype: str


### Задание 3

Примените преобразования к своей таблице `df` (из задания 2):

1. `cost_rub` → число.
2. `date` → дата, **правильным** способом. Проверьте, что все рейсы попали в май 2025.
3. `is_insured` → `bool` с пропусками там, где не заполнено.
4. Проверьте `df.dtypes`: остались ли столбцы с неподходящим типом?
5. Посчитайте суммарную стоимость рейсов — это ваша отправная точка для сравнения в конце пары.

In [185]:
insured_map = {"да": True, "1": True, "нет": False, "0": False}
# 1)
df["cost_rub"] = (df["cost_rub"].astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace("\u00a0", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float))

# 2)
d1 = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")
d2 = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
df["date"] = d1.fillna(d2)
not_may_2025 = df[(df["date"].dt.month!=5) | (df["date"].dt.year != 2025) | (df["date"].isna())]
print("рейсов не в мае 2025:", len(not_may_2025))
display(not_may_2025[["date", "trip_id"]])
# 3)
df["is_insured"] = (df["is_insured"].astype(str).str.strip().str.lower().map(insured_map).astype("boolean"))

display(df.head(15))
# 4)
print(df.dtypes)
# все столбцы с подходящим типом данных
# 5)
print("суммарная стоимость до чистки аномалий:", df["cost_rub"].sum())

рейсов не в мае 2025: 3


,date,trip_id
288,2026-05-04,T0009
63,2026-05-16,T0238
285,2026-05-10,T0270


,trip_id,date,route,carrier,cargo_type,client,weight_t,distance_km,duration_h,cost_rub,is_insured,delay_h
249,T0001,2025-05-17,Москва-Воронеж,волгатранс,зерно,НефтеТрейд,5.4,520,8.8,13763.35,False,0.0
258,T0002,2025-05-04,Санкт-Петербург-Петрозаводск,ТрансЛогистик,нефтепродукты,Агрохолдинг Нива,16.3,430,7.8,65234.80,True,8.0
9,T0003,2025-05-31,Москва-Воронеж,РусКарго,лес,НефтеТрейд,14.5,520,9.3,33118.73,False,2.0
146,T0004,2025-05-05,Екатеринбург-Челябинск,"АО ""СибПуть""",Стройматериалы,ЛесЭкспорт,21.0,220,3.6,27679.70,True,0.0
293,T0005,2025-05-23,Москва-Казань,сибпуть,Зерно,РитейлСеть,7.2,820,18.0,29602.15,False,NaN
315,T0006,2025-05-14,Москва-Нижний Новгород,ТРАНСЛОГИСТИК,зерно,Агрохолдинг Нива,3.3,420,7.0,7029.27,False,0.0
308,T0007,2025-05-23,Москва-Санкт-Петербург,транслогистик,Стройматериалы,СтройБаза 77,17.5,710,12.8,71106.47,True,4.0
489,T0008,2025-05-29,Екатеринбург-Челябинск,СибПуть,Зерно,НефтеТрейд,3.1,220,3.6,3399.28,<NA>,1.0
288,T0009,2026-05-04,Санкт-Петербург-Петрозаводск,РусКарго,МЕТАЛЛ,СтройБаза 77,10.2,430,9.1,31128.89,False,NaN
385,T0010,2025-05-14,Москва-Санкт-Петербург,ТРАНСЛОГИСТИК,Нефтепродукты,СтройБаза 77,16.5,710,12.4,102706.00,False,0.0


trip_id                   str
date           datetime64[us]
route                     str
carrier                   str
cargo_type                str
client                    str
weight_t              float64
distance_km             int64
duration_h            float64
cost_rub              float64
is_insured            boolean
delay_h               float64
dtype: object
суммарная стоимость до чистки аномалий: 20587629.76


---
# Часть 4. Нормализация значений

Одно и то же записано по-разному: `ТрансЛогистик`, `транслогистик`, `ООО "ТрансЛогистик"`,
`  ТрансЛогистик`. Для pandas это четыре разных перевозчика, и любая группировка развалится.

**Нормализация** — приведение значений к единому виду, который называют **каноническим**.

Базовые приёмы:

| Приём | Код |
|---|---|
| убрать пробелы по краям | `.str.strip()` |
| единый регистр | `.str.lower()` / `.str.capitalize()` |
| убрать двойные пробелы внутри | `.str.replace(r"\s+", " ", regex=True)` |

Но этого **недостаточно**: `ооо "транслогистик"` и `транслогистик` после приведения регистра
останутся разными. Юридическая форма — часть названия в одной системе и отсутствует в другой.
Поэтому нужен ещё один шаг: **справочник соответствий** (канонический словарь),
где для каждого варианта указано правильное имя.

In [186]:
# Шаг 1: базовая нормализация
step1 = raw["carrier"].str.strip().str.lower()
print("после strip+lower осталось вариантов:", step1.nunique())
print(sorted(step1.unique()))

после strip+lower осталось вариантов: 9
['автодорсервис', 'ао "сибпуть"', 'волгатранс', 'ооо "транслогистик"', 'ооо автодорсервис', 'ооо рускарго', 'рускарго', 'сибпуть', 'транслогистик']


In [187]:
# Шаг 2: убираем юридическую форму и приводим к канону по справочнику
canon = {
    "транслогистик": "ТрансЛогистик",
    "рускарго": "РусКарго",
    "сибпуть": "СибПуть",
    "волгатранс": "ВолгаТранс",
    "автодорсервис": "АвтоДорСервис",
}


def to_canon(value, canon_map):
    s = str(value).strip().lower()
    for prefix in ('ооо "', "ооо ", 'ао "', "ао "):      # убираем юридическую форму
        if s.startswith(prefix):
            s = s[len(prefix):]
    s = s.strip('"').strip()
    return canon_map.get(s, value)                       # не нашли — вернём как было


carrier_clean = raw["carrier"].apply(lambda v: to_canon(v, canon))
print("перевозчиков стало:", carrier_clean.nunique())
print(sorted(carrier_clean.unique()))
print()
print(carrier_clean.value_counts())

перевозчиков стало: 5
['АвтоДорСервис', 'ВолгаТранс', 'РусКарго', 'СибПуть', 'ТрансЛогистик']

carrier
РусКарго         109
ВолгаТранс        98
ТрансЛогистик     98
АвтоДорСервис     96
СибПуть           92
Name: count, dtype: int64


Обратите внимание на последнюю строку функции: если значение не нашлось в справочнике,
мы возвращаем его **как было**, а не `NaN`. Так неучтённый вариант останется видимым —
его легко найти, сравнив список уникальных значений с ожидаемым.

### Задание 4

1. Нормализуйте `carrier` в своей таблице. Проверьте: получилось ровно 5 перевозчиков?
2. Нормализуйте `cargo_type` — здесь достаточно `.str.strip().str.capitalize()`.
   Проверьте: 6 типов груза?
3. Нормализуйте `route`: уберите лишние пробелы. Проверьте: 8 маршрутов?
4. Проверьте `client`: сколько уникальных значений и нет ли там разных написаний?
5. **Контроль:** напишите проверку, которая сравнивает полученный набор перевозчиков
   с ожидаемым списком и печатает те, которых в списке нет.

In [188]:
# 1)
df["carrier"] = df["carrier"].apply(lambda x: to_canon(x, canon))
print("перевозчиков:", df["carrier"].nunique())

# 2)
df["cargo_type"] = df["cargo_type"].str.strip().str.capitalize()
print(df["cargo_type"].nunique())

# 3)
df["route"] = df["route"].str.strip().str.replace(r"\s+", " ", regex=True)
print(df["route"].nunique())
# 4)
print("клиентов:", df["client"].nunique())
print(df["client"].dropna().unique())
print()
# 5) контроль
expected_carriers = {"ТрансЛогистик", "РусКарго", "СибПуть", "ВолгаТранс", "АвтоДорСервис"}
proverka = set(df["carrier"].dropna().unique())
print(proverka - expected_carriers)

перевозчиков: 5
6
8
клиентов: 6
<StringArray>
[      'НефтеТрейд', 'Агрохолдинг Нива',       'ЛесЭкспорт',
       'РитейлСеть',     'СтройБаза 77',       'МеталлПром']
Length: 6, dtype: str

set()






---
# Часть 5. Пропуски

Главный вопрос про пропуск — **почему его нет**. От ответа зависит, что с ним делать.

| Столбец | Что означает пропуск | Разумное решение |
|---|---|---|
| `delay_h` | задержку не записали — но рейс был | подумать: `0` или «неизвестно»? |
| `weight_t` | вес не взвесили | нельзя придумать: заполнение искажает выручку за тонну |
| `client` | клиент не указан в заявке | оставить как есть, для выручки клиент не нужен |
| `is_insured` | признак не заполнен | третье состояние «неизвестно» — не `False`! |

**Разберите `delay_h` отдельно** — это показательный случай. Пропуск и ноль здесь значат
разное: `0` — «рейс пришёл вовремя», пропуск — «не знаем, записать забыли».
Если заполнить нулями, вы объявите, что все эти рейсы были без задержек, и доля задержек
в отчёте окажется ниже реальной.

Инструменты:

| Задача | Код |
|---|---|
| найти | `df["col"].isna().sum()` |
| удалить строки | `df.dropna(subset=["col"])` |
| заполнить значением | `df["col"].fillna(0)` |
| заполнить статистикой | `df["col"].fillna(df["col"].median())` |
| заполнить по группе | `df.groupby("route")["col"].transform("median")` |
| пометить флагом | `df["col_missing"] = df["col"].isna()` |

In [189]:
print("пропуски по столбцам:")
na = raw.isna().sum()
print(na[na > 0])
print()

# Сравним две стратегии для delay_h на доле задержек
delay = raw["delay_h"]
print("рейсов с записанной задержкой:", delay.notna().sum())
print("из них задержались:", (delay > 0).sum())
print()
print("доля задержек, если пропуски считать нулями:  {:.1%}".format(
    delay.fillna(0).gt(0).mean()))
print("доля задержек только по заполненным строкам:  {:.1%}".format(
    delay.dropna().gt(0).mean()))

пропуски по столбцам:
client        13
weight_t      29
is_insured    63
delay_h       40
dtype: int64

рейсов с записанной задержкой: 453
из них задержались: 128

доля задержек, если пропуски считать нулями:  26.0%
доля задержек только по заполненным строкам:  28.3%


Разница в несколько процентных пунктов — и это ещё мягкий случай. Выбор стратегии
меняет число в отчёте, поэтому он должен быть осознанным и записанным.

**Приём, который стоит взять в привычку:** заполняя пропуски, добавьте столбец-флаг.
Тогда позже вы всегда сможете отделить настоящие значения от заполненных.

In [190]:
demo = raw[["trip_id", "weight_t"]].head(0).copy()      # пустая заготовка для примера
demo = raw.loc[raw["weight_t"].isna(), ["trip_id", "weight_t"]].head(3).copy()
demo["weight_was_missing"] = demo["weight_t"].isna()
demo["weight_filled"] = demo["weight_t"].fillna(raw["weight_t"].median())
print(demo)
print()
print("медиана веса по всем рейсам:", raw["weight_t"].median())

   trip_id  weight_t  weight_was_missing  weight_filled
3    T0358       NaN                True           12.2
4    T0361       NaN                True           12.2
43   T0063       NaN                True           12.2

медиана веса по всем рейсам: 12.2


### Задание 5

По каждому столбцу с пропусками примите решение и примените его.

1. `delay_h`: выберите стратегию, обоснуйте, примените. Добавьте флаг `delay_missing`.
2. `weight_t`: **не заполняйте** средним — объясните в одном предложении, почему здесь
   это особенно вредно. (Подсказка: подумайте, что будет с показателем «стоимость за тонну».)
   Вместо этого добавьте флаг `weight_missing`.
3. `client`: оставьте как есть, но замените пропуски на понятную метку `"не указан"`.
4. `is_insured`: оставьте три состояния. Посчитайте, сколько рейсов в каждом.
5. Сведите решения в таблицу (markdown-ячейка ниже) — это часть отчёта о качестве данных.

In [191]:
# 1)
df["delay_missing"] = df["delay_h"].isna()
print(len(df[df["delay_h"].isna()]))
df["delay_h"] = df["delay_h"].fillna(df["delay_h"].median())
# заполняем медианной, чтобы примерно понимать время доставки
# 2)
df["weight_missing"] = df["weight_t"].isna()
print(len(df[df["weight_t"].isna()]))
# не заполняем, потому что может исказиться показатель "стоимость за тонну"
# 3)
print(len(df[df["client"].isna()]))
df["client"] = df["client"].fillna("не указан")
print()
# 4)
print("True", len(df[df["is_insured"] == True]))
print("False", len(df[df["is_insured"] == False]))
print("NA", len(df[df["is_insured"].isna()]))

40
29
13

True 210
False 207
NA 63


> **Мои решения по пропускам:**
>
> | Столбец | Пропусков | Решение | Почему |
> |---|---|---|---|
> | `delay_h` |40|Заполнить медианой + флаг|Заполняем медианной, чтобы примерно понимать время доставки|
> | `weight_t` |29|Оставить как есть + флаг|Не заполняем, потому что может исказиться показатель "стоимость за тонну"|
> | `client` |13|Заменить на "не указан"|Обозначаем, что нет данных|
> | `is_insured` |63|Оставляем как есть|NA - неизвестно, менять на True/False нельзя|

---
# Часть 6. Некорректные наблюдения и аномалии

Данные могут быть правильного типа, без пропусков и дублей — и при этом бессмысленными.
Три разных инструмента для трёх разных случаев:

| Тип проблемы | Пример | Чем ловить |
|---|---|---|
| **физически невозможное** | скорость 400 км/ч, вес −2.5 т, дата в 2026 году | знание предметной области: правила «так не бывает» |
| **статистический выброс** | рейс в 10 раз дороже обычного | `IQR`, границы распределения |
| **противоречие расчёту** | стоимость не соответствует тарифу и весу | сверка с независимо посчитанной величиной |

Начинайте всегда с первого: правила предметной области надёжнее любой статистики.
Фура не едет 400 км/ч — это не выброс, который «надо изучить», это ошибка в данных.

In [192]:
# Готовим рабочую копию с уже приведёнными типами
work = raw.copy()
work["cost_rub"] = (work["cost_rub"].astype(str).str.replace(" ", "", regex=False)
                    .str.replace("\u00a0", "", regex=False)
                    .str.replace(",", ".", regex=False).astype(float))
d1 = pd.to_datetime(work["date"], format="%d.%m.%Y", errors="coerce")
d2 = pd.to_datetime(work["date"], format="%Y-%m-%d", errors="coerce")
work["date"] = d1.fillna(d2)

# 1. Физически невозможное
work["speed_kmh"] = (work["distance_km"] / work["duration_h"]).round(1)

print("максимальная скорость в данных:", work["speed_kmh"].max(), "км/ч")
print("рейсов со скоростью выше 110 км/ч:", (work["speed_kmh"] > 110).sum())
print("рейсов с весом <= 0:", (work["weight_t"] <= 0).sum())
print("рейсов с датой позже 31.05.2025:", (work["date"] > "2025-05-31").sum())
print()
print(work[work["speed_kmh"] > 110][["trip_id", "route", "distance_km", "duration_h", "speed_kmh"]].head())

максимальная скорость в данных: 400.0 км/ч
рейсов со скоростью выше 110 км/ч: 9
рейсов с весом <= 0: 9
рейсов с датой позже 31.05.2025: 3

    trip_id                         route  distance_km  duration_h  speed_kmh
86    T0191  Санкт-Петербург-Петрозаводск          430         1.5      286.7
123   T0233                Москва-Воронеж          520         1.4      371.4
180   T0023                Москва-Казань           820         3.2      256.2
201   T0148                 Москва-Казань          820         3.4      241.2
219   T0135        Москва-Нижний Новгород          420         1.1      381.8


In [193]:
# 2. Статистический выброс: метод межквартильного размаха (IQR)
q1 = work["cost_rub"].quantile(0.25)
q3 = work["cost_rub"].quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 = {q1:,.0f} | Q3 = {q3:,.0f} | IQR = {iqr:,.0f}".replace(",", " "))
print(f"границы: от {low:,.0f} до {high:,.0f}".replace(",", " "))
print("выбросов по IQR:", ((work["cost_rub"] < low) | (work["cost_rub"] > high)).sum())
print()
print("обратите внимание: нижняя граница отрицательная —")
print("для стоимости это бессмысленно, снизу метод здесь ничего не даёт")

Q1 = 19 614 | Q3 = 58 426 | IQR = 38 811
границы: от -38 602 до 116 643
выбросов по IQR: 12

обратите внимание: нижняя граница отрицательная —
для стоимости это бессмысленно, снизу метод здесь ничего не даёт


In [194]:
# 3. Противоречие расчёту: сверяем стоимость с тарифом
tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}

work["cargo_clean"] = work["cargo_type"].str.strip().str.capitalize()
work["tariff"] = work["cargo_clean"].map(tariffs)
work["expected_cost"] = work["weight_t"] * work["distance_km"] * work["tariff"]
work["ratio"] = (work["cost_rub"] / work["expected_cost"]).round(2)

print("рейсов, где стоимость завышена более чем в 5 раз:", (work["ratio"] > 5).sum())
print()
print(work[work["ratio"] > 5][["trip_id", "cargo_clean", "weight_t", "distance_km",
                               "cost_rub", "expected_cost", "ratio"]].head())

рейсов, где стоимость завышена более чем в 5 раз: 5

    trip_id     cargo_clean  weight_t  distance_km   cost_rub  expected_cost  \
59    T0111          Металл       0.0          820  127381.55            0.0   
346   T0100             Лес       0.0          360   14110.32            0.0   
411   T0149       Контейнер       0.0          520   75975.90            0.0   
423   T0371           Зерно       0.0          430   42463.82            0.0   
450   T0263  Стройматериалы       5.0          820  230546.04        24190.0   

     ratio  
59     inf  
346    inf  
411    inf  
423    inf  
450   9.53  


**Важно: выброс — это не то же самое, что ошибка.**

IQR нашёл 12 «выбросов» по стоимости. Но дорогой рейс — это нормально: 22 тонны
нефтепродуктов на 820 км действительно стоят много. Из этих 12 ошибочны только те,
где стоимость противоречит тарифу.

Отсюда правило: статистический метод **показывает, что посмотреть**, а решение
«ошибка или нет» принимается по смыслу, а не по формуле. Удалять всё, что вышло
за границы IQR, — верный способ выбросить из отчёта самых крупных клиентов.

Ещё одна деталь: при делении на нулевой вес получается `inf`, а не ошибка.
Такие значения тоже нужно учитывать — проверяйте через `np.isinf()`.

### Задание 6

1. Найдите в своей таблице все **физически невозможные** наблюдения: скорость выше 110 км/ч,
   вес не больше нуля, дата вне мая 2025. Сколько таких строк всего?
2. Посчитайте границы IQR для `cost_rub` и число выбросов.
3. Сверьте стоимость с расчётом по тарифу. Сколько рейсов отличаются более чем в 5 раз?
   Сколько из них попали и в выбросы по IQR?
4. **Примите решения:** какие строки удалить, какие оставить с пометкой. Создайте
   финальную таблицу `clean`.
5. Посчитайте, сколько строк удалено и какую долю от исходных данных это составляет.

In [195]:
import numpy as np

# 1)
bad_speed = (df["distance_km"] / df["duration_h"]) > 110
bad_weight = df["weight_t"] <= 0
bad_date = ((df["date"].dt.month != 5) | (df["date"].dt.year != 2025))
all_bad = bad_speed | bad_weight | bad_date
print("невозможных наблюдений:", len(df[all_bad]))
print()
# 2)
q1 = df["cost_rub"].quantile(0.25)
q3 = df["cost_rub"].quantile(0.75)
iqr = q3 - q1
low, high = q1 -  1.5 * iqr, q3 + 1.5 * iqr
vibros = (df["cost_rub"] < low) | (df["cost_rub"] > high)
print(vibros.sum())
print()
# 3)
cargo_clean= df["cargo_type"].str.strip().str.capitalize()
tariff = cargo_clean.map(tariffs)
expected_cost = df["weight_t"] * df["distance_km"] * tariff
otnoshenie = df["cost_rub"] / expected_cost
bad_cost = otnoshenie > 5
print("Отличаются более чем в 5 раз:", bad_cost.sum())
print("Попали и в выбросы:", (bad_cost & vibros).sum())
print()
# 4)
# удаляем строки с физически невозможными данными и явными ошибками в стоимости
clean = df[~all_bad & ~bad_cost].copy()
# 5)
print("удалено строк:", len(df)-len(clean), "доля:", ((len(df)-len(clean))/len(df)*100))

невозможных наблюдений: 21

12

Отличаются более чем в 5 раз: 5
Попали и в выбросы: 2

удалено строк: 22 доля: 4.583333333333333


---
# Часть 7. Мини-проект: пайплайн очистки и отчёт о качестве

Теперь соберите всё в одну функцию. Это то, как очистка живёт в реальных проектах:
не разбросанные по ноутбуку ячейки, а один воспроизводимый конвейер, который можно
запустить на новой выгрузке в следующем месяце.

### Задание 7

Напишите функцию `clean_trips(raw)`, которая делает всё по порядку и возвращает
**две** вещи: очищенную таблицу и отчёт о качестве (словарь или `DataFrame`).

Порядок шагов важен — он не случаен:

```
1. дубликаты            (чем меньше строк, тем быстрее остальное)
2. типы                 (без чисел и дат нельзя проверять значения)
3. нормализация текста   (иначе развалятся группировки и подстановка тарифа)
4. пропуски             (решения записаны в части 5)
5. аномалии             (нужны и типы, и нормализованный груз для тарифа)
```

В отчёт о качестве включите: сколько строк пришло, сколько удалено на каждом шаге,
сколько осталось, доля потерь и итоговая сумма `cost_rub`.

In [196]:
def clean_trips(raw):
    report = {"строк получено": len(raw)}
    df = raw.copy()
    
    # 1. дубликаты
    df = df.drop_duplicates()
    df["cost_num"] = df["cost_rub"].str.replace(" ", "").str.replace(",", ".").astype(float)
    df = df.sort_values(by=["trip_id", "cost_num"], ascending=[True, False])
    df = df.drop_duplicates(subset="trip_id", keep="first")
    df = df.drop(columns=["cost_num"])

    # 2. типы
    df["cost_rub"] = df["cost_rub"].str.replace(" ", "", regex=False).str.replace("\u00a0", "", regex=False).str.replace(",", ".", regex=False).astype(float)
    d1 = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")
    d2 = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
    df["date"] = d1.fillna(d2)
    insured_map = {"да": True, "1": True, "нет": False, "0": False}
    df["is_insured"] = (df["is_insured"].astype(str).str.strip().str.lower().map(insured_map).astype("boolean"))

    # 3. нормализация
    df["carrier"] = df["carrier"].apply(lambda x: to_canon(x, canon))
    df["cargo_type"] = df["cargo_type"].str.strip().str.capitalize()
    df["route"] = df["route"].str.strip().str.replace(r"\s+", " ", regex=True)

    # 4. пропуски
    df["delay_missing"] = df["delay_h"].isna()
    df["delay_h"] = df["delay_h"].fillna(df["delay_h"].median())
    df["weight_missing"] = df["weight_t"].isna()
    df["client"] = df["client"].fillna("не указан")

    # 5. аномалии
    bad_speed = (df["distance_km"] / df["duration_h"]) > 110
    bad_weight = df["weight_t"] <= 0
    bad_date = ((df["date"].dt.month != 5) | (df["date"].dt.year != 2025))
    tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}
    tariff = df["cargo_type"].map(tariffs)
    expected_cost = df["weight_t"] * df["distance_km"] * tariff
    otnoshenie = df["cost_rub"] / expected_cost
    bad_cost = otnoshenie > 5
    all_bad = bad_speed | bad_weight | bad_date | bad_cost
    clean = df[~all_bad].copy() 

    report["удалено строк"] = len(raw) - len(clean)
    report["строк осталось"] = len(clean)
    report["доля потерь"] = report["удалено строк"] / len(raw) * 100
    report["итоговая сумма"] = clean["cost_rub"].sum()
    return clean, report


clean, report = clean_trips(raw)

print("ОТЧЁТ О КАЧЕСТВЕ ДАННЫХ")
print("-" * 44)
for k, v in report.items():
    print(f"{k:<28} {v}")

ОТЧЁТ О КАЧЕСТВЕ ДАННЫХ
--------------------------------------------
строк получено               493
удалено строк                35
строк осталось               458
доля потерь                  7.099391480730223
итоговая сумма               19320777.650000002


### Задание 8. Цена очистки

Главный результат пары — не очищенная таблица, а ответ на вопрос «насколько отчёт
был бы неверным без очистки».

1. Посчитайте суммарную стоимость рейсов **по сырым данным** (только приведя тип, без
   удаления дублей и аномалий).
2. Посчитайте суммарную стоимость по очищенным данным.
3. На сколько рублей и на сколько процентов они расходятся?
4. Посчитайте долю рейсов с задержкой в двух вариантах: по сырым данным и по очищенным.
5. Напишите три предложения для руководителя: какие данные вы получили, что с ними
   сделали и каким числам теперь можно верить.

In [197]:
# 1) - 4)
raw_cost = raw["cost_rub"].astype(str).str.replace(" ", "").str.replace(",", ".").astype(float).sum()
clean_cost = report["итоговая сумма"]
diff_rub = raw_cost - clean_cost
diff_procent = (diff_rub / raw_cost) * 100
raw_delay = (raw["delay_h"] > 0).sum() / len(raw) * 100
clean_delay = (clean["delay_h"] > 0).sum() / len(clean) * 100
print("1", raw_cost)
print("2", clean_cost)
print("3", diff_rub, ", в %:", diff_procent)
print("4", raw_delay, clean_delay)

1 21045802.360000003
2 19320777.650000002
3 1725024.710000001 , в %: 8.19652622642989
4 25.963488843813387 26.200873362445414


> **Для руководителя:**
>
> 1. Получена выгрузка на 493 строки. В ней были обнаружены технические дубли, разные форматы дат и нереалистичные данные (например, слишком высокая скорость)
> 2. Удалено 35 бракованных строк. Оставшиеся 458 записей были очищены от аномалий, дубликатов, а названия перевозчиков были приведены к единому стандарту
> 3. Исходный отчёт завышал бы выручку из-за мусора в данных. Но очищенным данным, где выручка составила 19320778 руб, можно верить

---
## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 8: очистка данных и отчёт о качестве"
git push
```

## Чек-лист сдачи

- [ ] Задание 1: карта проблем заполнена числами
- [ ] Задание 2: три вида дубликатов различены, решение по конфликтующим обосновано
- [ ] Задание 3: типы приведены, даты разобраны **правильным** способом, все рейсы в мае
- [ ] Задание 4: 5 перевозчиков, 6 типов груза, 8 маршрутов, есть проверка на неучтённые значения
- [ ] Задание 5: по каждому столбцу с пропусками записано решение и причина
- [ ] Задание 6: невозможные значения, IQR и сверка с тарифом разделены
- [ ] Задание 7: пайплайн собран в функцию, отчёт о качестве печатается
- [ ] Задание 8: посчитана разница в выручке до и после очистки
- [ ] Всё запушено в репозиторий

## Что унести с пары

Очистка данных не имеет «правильного ответа», который можно списать. Заполнить пропуск
нулём или оставить, удалить выброс или пометить — это **решения**, и они меняют итоговые
числа. Профессионализм здесь не в том, чтобы знать методы (их десяток), а в том, чтобы
каждое решение осознавать, записывать и уметь объяснить.

И главное: данные, прошедшие проверку типов, пропусков и дублей, всё ещё могут быть
бессмысленными. Фура со скоростью 400 км/ч — корректная строка с точки зрения pandas.
Поэтому последний контур проверки — всегда знание предметной области.